# 03_03 - Data Integration: Weather & OMIP Financials

## 1. Objective
This notebook executes the relational join between our cleaned meteorological environment and the OMIP financial time-series. This merge creates the **Consolidated State-Space** that our Reinforcement Learning agent will use to learn correlations between weather shocks and price volatility.

**Join Strategy: Inner Join vs. Left Join**
- **Weather Coverage:** Continuous daily data (365 days/year).
- **Financial Coverage:** Business days only (spot prices are daily, but futures trading often stops on weekends/holidays).

We will perform an **Inner Join** based on the `date` column. This ensures that every observation in our final dataset has both a target (Price) and its corresponding features (Weather), eliminating any "blind" rows that would confuse the agent during training.

In [1]:
import pandas as pd
import sys
import os
from pathlib import Path

# Setup paths
project_root = Path(os.path.abspath('../../'))
weather_path = project_root / "data" / "interim" / "weather_clean.csv"
omip_path = project_root / "data" / "interim" / "omip_clean.csv"
output_path = project_root / "data" / "interim" / "merged_energy_weather.csv"

# 1. Load cleaned datasets
df_weather = pd.read_csv(weather_path)
df_omip = pd.read_csv(omip_path)

# Ensure datetime format
df_weather['date'] = pd.to_datetime(df_weather['date'])
df_omip['date'] = pd.to_datetime(df_omip['date'])

print(f"Weather Dataset: {df_weather.shape} rows")
print(f"OMIP Dataset: {df_omip.shape} rows")

# 2. Execute the Merge (Inner Join on 'date')
df_merged = pd.merge(df_omip, df_weather, on='date', how='inner')

print(f"\n✅ Merge Complete. Consolidated Shape: {df_merged.shape}")

Weather Dataset: (2275, 21) rows
OMIP Dataset: (2192, 14) rows

✅ Merge Complete. Consolidated Shape: (2192, 34)


## 2. Post-Merge Integrity Audit

After merging, we must verify if the temporal alignment caused any unintended data loss and check the final feature count.

In [2]:
# Check for nulls introduced (should be zero if inner join worked correctly)
nulls = df_merged.isnull().sum().sum()
print(f"Total Missing Values in Merged Set: {nulls}")

# Check Temporal Continuity
date_range = (df_merged['date'].max() - df_merged['date'].min()).days + 1
print(f"Temporal Span: {df_merged['date'].min().date()} to {df_merged['date'].max().date()}")
print(f"Theoretical days in span: {date_range}")
print(f"Actual rows: {len(df_merged)}")

# Display Sample
display(df_merged[['date', 'Spot_Price_SPEL', 'temperature_2m_mean', 'wind_speed_10m_max']].head())

Total Missing Values in Merged Set: 12
Temporal Span: 2020-01-01 to 2025-12-31
Theoretical days in span: 2192
Actual rows: 2192


,date,Spot_Price_SPEL,temperature_2m_mean,wind_speed_10m_max
0,2020-01-01,35.54,6.535541,9.113214
1,2020-01-02,40.00,6.054612,9.889774
2,2020-01-03,39.51,6.174448,11.881113
3,2020-01-04,35.67,6.525313,13.375087
4,2020-01-05,38.18,6.535853,11.201827


In [3]:
# Save the master dataset
output_path.parent.mkdir(parents=True, exist_ok=True)
df_merged.to_csv(output_path, index=False)
print(f"✅ Master Consolidated Dataset saved to:\n{output_path}")

✅ Master Consolidated Dataset saved to:
c:\Users\Alejandro\GitHub\Data-Driven-Strategies-for-Financial-Resilience-in-Energy-Procurement\data\interim\merged_energy_weather.csv
